Criação do workpace / servidor

In [0]:
%sql
create database if not exists workspace.engenharia_de_dados
COMMENT 'Projeto acadêmico — PUC RIO - Dados do Censo Demográfico 2022 (IBGE)'

Importação de dados

In [0]:
%pip install google-cloud-bigquery

In [0]:
dbutils.library.restartPython()

In [0]:
from google.cloud import bigquery
import pandas as pd

In [0]:
%pip install basedosdados

In [0]:
import json
from google.cloud import bigquery
from google.oauth2 import service_account


# Recupera a credencial do Databricks Secret Scope
service_account_key = dbutils.secrets.get(
    scope="gcp-secrets",
    key="gcp-service-account-json"
)

# Cria as credenciais diretamente em memória
credentials = service_account.Credentials.from_service_account_info(
    json.loads(service_account_key)
)

# Cria o cliente BigQuery
client = bigquery.Client(
    credentials=credentials,
    project=credentials.project_id
)
 ## Queries para ingestão das bases (camada bronze)


query_mun = """
SELECT
   *
FROM `basedosdados.br_ibge_censo_2022.municipio`
WHERE sigla_uf = 'RJ'
"""
query_dados = """
SELECT
    
    dados.id_municipio AS id_municipio,
    dados.id_setor_censitario as id_setor_censitario,
    dados.area as area,
    dados.pessoas as pessoas,
    dados.domicilios_particulares as domicilios_particulares,
    dados.media_moradores_domicilios as media_moradores_domicilios,
 
FROM `basedosdados.br_ibge_censo_2022.setor_censitario` AS dados
LEFT JOIN (SELECT DISTINCT *  FROM `basedosdados.br_ibge_censo_2022.municipio`) AS dic_municipio
    ON dados.id_municipio = dic_municipio.id_municipio
WHERE sigla_uf = 'RJ'
"""


## Criação dos dataframes
dim_municipio = client.query(query_mun).to_dataframe(
    create_bqstorage_client=False
)
dados = client.query(query_dados).to_dataframe(
    create_bqstorage_client=False
)




In [0]:
# Visualização rápida dos dados (amostra)
dim_municipio.head()


In [0]:
dados.head()